# UC06 — Predicción de Fuga en Renovación

Modelo predictivo de no-renovación 90 días antes del vencimiento.
**Valor de negocio**: Activar campañas de retención dirigidas antes de perder al cliente.

---

## Paso 1: Configuración del Entorno

In [ ]:
CREATE DATABASE IF NOT EXISTS RETENCION_DB;
USE SCHEMA RETENCION_DB.PUBLIC;
CREATE WAREHOUSE IF NOT EXISTS COMPUTE_WH WITH WAREHOUSE_SIZE='SMALL' AUTO_SUSPEND=60 AUTO_RESUME=TRUE;
USE WAREHOUSE COMPUTE_WH;

---

## Paso 2: Generar Cartera de Pólizas

In [ ]:
CREATE OR REPLACE TABLE cartera AS
SELECT 'POL' || LPAD(SEQ4(),5,'0') AS poliza_id,
    CASE MOD(SEQ4(),4) WHEN 0 THEN 'Auto' WHEN 1 THEN 'Hogar' WHEN 2 THEN 'Vida' ELSE 'Salud' END AS producto,
    UNIFORM(200,3000,RANDOM())::DECIMAL(10,2) AS prima_anual, UNIFORM(1,20,RANDOM()) AS anos_renovando,
    DATEADD(day,UNIFORM(1,90,RANDOM()),CURRENT_DATE()) AS fecha_vencimiento,
    UNIFORM(0,3,RANDOM()) AS siniestros_ultimo_ano,
    CASE WHEN UNIFORM(0,1,RANDOM())<0.4 THEN 1 ELSE 0 END AS tiene_descuento,
    CASE MOD(UNIFORM(1,100,RANDOM()),3) WHEN 0 THEN 'Online' WHEN 1 THEN 'Agente' ELSE 'Broker' END AS canal,
    CASE WHEN UNIFORM(0,1,RANDOM())<0.25 THEN 1 ELSE 0 END AS no_renueva
FROM TABLE(GENERATOR(ROWCOUNT=>1500));

---

## Paso 3: Historial de Interacciones

In [ ]:
CREATE OR REPLACE TABLE interacciones AS
SELECT c.poliza_id,
    CASE WHEN c.no_renueva=1 THEN UNIFORM(3,10,RANDOM()) ELSE UNIFORM(0,3,RANDOM()) END AS llamadas_90d,
    CASE WHEN c.no_renueva=1 THEN UNIFORM(1,5,RANDOM()) ELSE UNIFORM(0,1,RANDOM()) END AS quejas_90d,
    CASE WHEN c.no_renueva=1 AND UNIFORM(0,1,RANDOM())<0.6 THEN 1 ELSE 0 END AS ha_pedido_competencia,
    CASE WHEN c.no_renueva=1 THEN UNIFORM(1,5,RANDOM()) ELSE UNIFORM(6,10,RANDOM()) END AS nps,
    CASE WHEN c.no_renueva=1 THEN UNIFORM(5,30,RANDOM()) ELSE UNIFORM(40,95,RANDOM()) END::FLOAT AS emails_abiertos_pct,
    CASE WHEN c.no_renueva=1 AND UNIFORM(0,1,RANDOM())<0.7 THEN 0 ELSE 1 END AS acceso_area_cliente_30d
FROM cartera c;

---

## Paso 4: Feature Engineering

In [ ]:
CREATE OR REPLACE TABLE features_fuga AS
SELECT c.poliza_id, c.prima_anual::FLOAT AS prima, c.anos_renovando::FLOAT AS anos_renovando,
    c.siniestros_ultimo_ano::FLOAT AS siniestros, c.tiene_descuento::FLOAT AS tiene_descuento,
    CASE WHEN c.canal='Online' THEN 1.0 ELSE 0.0 END AS canal_online,
    CASE WHEN c.canal='Agente' THEN 1.0 ELSE 0.0 END AS canal_agente,
    i.llamadas_90d::FLOAT AS llamadas, i.quejas_90d::FLOAT AS quejas,
    i.ha_pedido_competencia::FLOAT AS pedido_competencia, i.nps::FLOAT AS nps,
    i.emails_abiertos_pct AS emails_pct, i.acceso_area_cliente_30d::FLOAT AS acceso_web,
    c.no_renueva
FROM cartera c JOIN interacciones i ON c.poliza_id=i.poliza_id;

---

## Paso 5: Train/Test

In [ ]:
CREATE OR REPLACE TABLE entrenamiento AS SELECT * FROM features_fuga SAMPLE(80);
CREATE OR REPLACE TABLE test AS SELECT * FROM features_fuga WHERE poliza_id NOT IN (SELECT poliza_id FROM entrenamiento);

---

## Paso 6: Entrenar Modelo

In [ ]:
CREATE OR REPLACE SNOWFLAKE.ML.CLASSIFICATION predictor_fuga(
    INPUT_DATA=>SYSTEM$REFERENCE('TABLE','entrenamiento'),
    TARGET_COLNAME=>'no_renueva',
    CONFIG_OBJECT=>{'evaluate':TRUE}
);

---

## Paso 7: Evaluar

In [ ]:
CALL predictor_fuga!SHOW_EVALUATION_METRICS();

In [ ]:
CALL predictor_fuga!SHOW_FEATURE_IMPORTANCE();

---

## Paso 8: Puntuar y Priorizar

In [ ]:
CREATE OR REPLACE TABLE predicciones_fuga AS
SELECT poliza_id,
    predictor_fuga!PREDICT(OBJECT_CONSTRUCT(
        'prima',prima,'anos_renovando',anos_renovando,'siniestros',siniestros,'tiene_descuento',tiene_descuento,
        'canal_online',canal_online,'canal_agente',canal_agente,'llamadas',llamadas,'quejas',quejas,
        'pedido_competencia',pedido_competencia,'nps',nps,'emails_pct',emails_pct,'acceso_web',acceso_web
    )) AS prediccion,
    prediccion['probability']['1']::FLOAT AS prob_fuga,
    CASE WHEN prediccion['probability']['1']::FLOAT>=0.70 THEN 'Fuga Muy Probable'
         WHEN prediccion['probability']['1']::FLOAT>=0.40 THEN 'Riesgo Moderado'
         ELSE 'Probable Renovación' END AS nivel,
    prima, no_renueva AS fuga_real
FROM test;

SELECT * FROM predicciones_fuga ORDER BY prob_fuga DESC LIMIT 20;

---

## Paso 9: Dashboard Interactivo

In [ ]:
import streamlit as st
import pandas as pd
from snowflake.snowpark.context import get_active_session

session = get_active_session()
st.title('Predicción de Fuga en Renovación')
df = session.table('predicciones_fuga').to_pandas()

c1,c2,c3,c4 = st.columns(4)
with c1: st.metric('Pólizas Analizadas', len(df))
with c2: st.metric('Fuga Muy Probable', len(df[df['NIVEL']=='Fuga Muy Probable']))
with c3: st.metric('Prima en Riesgo', f"{df[df['NIVEL']=='Fuga Muy Probable']['PRIMA'].sum():,.0f}€")
with c4: st.metric('Exactitud', f"{(df['PROB_FUGA'].round()==df['FUGA_REAL']).mean():.1%}")

st.markdown('---')
rc = df['NIVEL'].value_counts()
st.bar_chart(pd.DataFrame({'Pólizas': rc.values}, index=rc.index))

st.markdown('---')
st.subheader('Top Pólizas para Retención')
top = df[df['NIVEL']=='Fuga Muy Probable'].nlargest(20,'PRIMA')
top['Prob %'] = top['PROB_FUGA'].apply(lambda x: f'{x:.1%}')
st.dataframe(top[['POLIZA_ID','Prob %','PRIMA','FUGA_REAL']], use_container_width=True)
st.caption('Desarrollado con Snowflake Cortex ML + Streamlit')

---

## Paso 10: Limpieza (Opcional)

In [ ]:
-- DROP DATABASE IF EXISTS RETENCION_DB;
-- DROP WAREHOUSE IF EXISTS COMPUTE_WH;